# Data Formats & Storage — 08: Avro + Schema Registry Pattern

## What you will learn
Avro is the standard format for **event streaming** pipelines.
Unlike Parquet (optimised for batch reads), Avro is optimised for:
- Row-by-row serialization (ideal for Kafka messages)
- Schema evolution with backward/forward compatibility
- Schema registry integration

In this notebook you will:
1. Understand Avro's binary format and schema definition
2. Write and read Avro files with Spark
3. Implement schema evolution (adding, removing, renaming fields)
4. Simulate a Schema Registry pattern for Kafka-like pipelines
5. Benchmark Avro vs Parquet for streaming vs batch workloads
6. Build a realistic Kafka → Avro → Parquet pipeline simulation

## Avro vs Parquet: When to Use Which

```
Avro (row-based):
  ┌────────────────────────────────┐
  │ schema (JSON header)           │
  ├────────────────────────────────┤
  │ row 1: [id=1, name=Alice, ...] │
  │ row 2: [id=2, name=Bob,   ...] │
  │ row 3: [id=3, name=Carol, ...] │
  └────────────────────────────────┘
  Best for: streaming (Kafka), row-level access, schema evolution

Parquet (columnar):
  ┌───────┬───────────┬──────────┐
  │ id    │ name      │ salary   │
  │ 1,2,3 │ Alice,Bob │ 90K,72K  │
  └───────┴───────────┴──────────┘
  Best for: analytics, aggregations, column pruning
```

**Rule of thumb:** Use Avro for transport (Kafka), convert to Parquet for storage.


In [ ]:
import os, time, random, datetime, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('avro-schema-registry')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version}")

## Step 1 — Avro Schema Definition

Avro schemas are defined in JSON. The schema is stored WITH the data
(in the file header for Avro files, or in a Schema Registry for Kafka).


In [ ]:
import json as pyjson, pathlib, shutil, time, random, datetime

DATA_DIR = '/workspace/data'
AVRO_DIR = f'{DATA_DIR}/avro_demo'
shutil.rmtree(AVRO_DIR, ignore_errors=True)
pathlib.Path(AVRO_DIR).mkdir(parents=True, exist_ok=True)

# Avro schema v1 — initial event schema
SCHEMA_V1 = {
    "type": "record",
    "name": "UserEvent",
    "namespace": "com.example.events",
    "doc": "User interaction event from web application",
    "fields": [
        {"name": "event_id",    "type": "long",   "doc": "Unique event ID"},
        {"name": "user_id",     "type": "long",   "doc": "User identifier"},
        {"name": "event_type",  "type": "string", "doc": "Type: click/view/purchase/search"},
        {"name": "page",        "type": "string", "doc": "Page URL"},
        {"name": "revenue",     "type": "double", "doc": "Revenue if purchase, else 0.0"},
        {"name": "event_ts",    "type": "long",   "doc": "Epoch milliseconds"},
    ]
}

print("Avro Schema v1:")
print(pyjson.dumps(SCHEMA_V1, indent=2))
print()
print("Key Avro types: null, boolean, int, long, float, double, bytes, string")
print("Complex types: record, enum, array, map, union, fixed")

## Step 2 — Write and Read Avro with Spark


In [ ]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

random.seed(42)
N = 200_000
EVENTS    = ["click","view","purchase","search","add_to_cart"]
PAGES     = ["/home","/products","/cart","/checkout","/search"]

# Generate events matching schema v1
events_df = spark.range(N).select(
    F.col("id").alias("event_id"),
    (F.col("id") % 50_000 + 1).alias("user_id"),
    F.element_at(F.array([F.lit(e) for e in EVENTS]),
                 (F.col("id") % 5 + 1).cast("int")).alias("event_type"),
    F.element_at(F.array([F.lit(p) for p in PAGES]),
                 (F.col("id") % 5 + 1).cast("int")).alias("page"),
    F.when(F.col("id") % 7 == 0, F.round(F.rand(42)*500+10, 2)).otherwise(F.lit(0.0)).alias("revenue"),
    (F.unix_timestamp(F.current_timestamp()) * 1000 - F.col("id")).alias("event_ts"),
)

# Write Avro
AVRO_V1 = f"{AVRO_DIR}/events_v1"
t0 = time.time()
events_df.write.format("avro").mode("overwrite").save(AVRO_V1)
t_write = time.time() - t0

# Read Avro
t0 = time.time()
read_back = spark.read.format("avro").load(AVRO_V1)
count = read_back.count()
t_read = time.time() - t0

print(f"Avro write: {t_write:.2f}s | {count:,} rows")
print(f"Avro read : {t_read:.3f}s")
print()
read_back.printSchema()
read_back.show(5)

## Step 3 — Schema Evolution

Avro's most important feature: schemas can evolve while maintaining compatibility.

**Backward compatibility**: new schema can read old data
**Forward compatibility**: old schema can read new data
**Full compatibility**: both directions work

Rules:
- Adding a field with a **default value** → backward compatible ✅
- Removing a field → forward compatible ✅ (old readers ignore unknown fields)
- Renaming a field → breaking change ❌ (use aliases)
- Changing a type → usually breaking ❌


In [ ]:
# Schema v2: adds two new fields WITH defaults (backward compatible)
SCHEMA_V2 = {
    "type": "record",
    "name": "UserEvent",
    "namespace": "com.example.events",
    "fields": [
        {"name": "event_id",    "type": "long"},
        {"name": "user_id",     "type": "long"},
        {"name": "event_type",  "type": "string"},
        {"name": "page",        "type": "string"},
        {"name": "revenue",     "type": "double"},
        {"name": "event_ts",    "type": "long"},
        # New fields with defaults — backward compatible
        {"name": "device",      "type": "string",  "default": "unknown",
         "doc": "Device type: mobile/desktop/tablet"},
        {"name": "country",     "type": "string",  "default": "XX",
         "doc": "ISO country code"},
        {"name": "session_id",  "type": ["null","string"], "default": None,
         "doc": "Session identifier (nullable union)"},
    ]
}

DEVICES   = ["mobile","desktop","tablet"]
COUNTRIES = ["US","UK","DE","FR","JP","CA"]

# Write v2 events (new schema with new fields populated)
events_v2 = spark.range(N, N*2).select(
    F.col("id").alias("event_id"),
    (F.col("id") % 50_000 + 1).alias("user_id"),
    F.element_at(F.array([F.lit(e) for e in EVENTS]),
                 (F.col("id") % 5 + 1).cast("int")).alias("event_type"),
    F.element_at(F.array([F.lit(p) for p in PAGES]),
                 (F.col("id") % 5 + 1).cast("int")).alias("page"),
    F.when(F.col("id") % 7 == 0, F.round(F.rand(42)*500+10, 2)).otherwise(F.lit(0.0)).alias("revenue"),
    (F.unix_timestamp(F.current_timestamp()) * 1000 - F.col("id")).alias("event_ts"),
    F.element_at(F.array([F.lit(d) for d in DEVICES]),
                 (F.col("id") % 3 + 1).cast("int")).alias("device"),
    F.element_at(F.array([F.lit(c) for c in COUNTRIES]),
                 (F.col("id") % 6 + 1).cast("int")).alias("country"),
    F.concat(F.lit("sess_"), (F.col("id") % 5000).cast("string")).alias("session_id"),
)

AVRO_V2 = f"{AVRO_DIR}/events_v2"
events_v2.write.format("avro").mode("overwrite").save(AVRO_V2)
print("v2 events written with new fields: device, country, session_id")

# Read v1 data with v2 reader — old rows get default values for new fields
v1_with_v2_schema = spark.read.format("avro").load(AVRO_V1)
print()
print("v1 data read (no device/country/session_id — shows null):")
v1_with_v2_schema.select("event_id","event_type","revenue").show(3)

print("v2 data (has device/country/session_id):")
spark.read.format("avro").load(AVRO_V2).select(
    "event_id","event_type","device","country","session_id").show(3)

## Step 4 — Schema Registry Pattern

In Kafka pipelines, schemas are stored in a **Schema Registry** (e.g., Confluent).
Each message carries only a schema ID (4 bytes), not the full schema.
The consumer looks up the schema by ID to deserialize.

We simulate this pattern using a local file as our "registry".


In [ ]:
# Simulate Schema Registry as a local JSON file
REGISTRY_PATH = f"{AVRO_DIR}/schema_registry.json"

def registry_load():
    try:
        return pyjson.loads(pathlib.Path(REGISTRY_PATH).read_text())
    except:
        return {"schemas": {}, "subjects": {}}

def registry_register(subject, schema):
    """Register a schema, return schema_id."""
    reg = registry_load()
    # Check if schema already registered
    schema_str = pyjson.dumps(schema, sort_keys=True)
    for sid, s in reg["schemas"].items():
        if pyjson.dumps(s, sort_keys=True) == schema_str:
            return int(sid)
    # New schema
    new_id = max([int(k) for k in reg["schemas"].keys()], default=0) + 1
    reg["schemas"][str(new_id)] = schema
    if subject not in reg["subjects"]:
        reg["subjects"][subject] = []
    reg["subjects"][subject].append(new_id)
    pathlib.Path(REGISTRY_PATH).write_text(pyjson.dumps(reg, indent=2))
    return new_id

def registry_get(schema_id):
    reg = registry_load()
    return reg["schemas"].get(str(schema_id))

def registry_latest(subject):
    reg = registry_load()
    ids = reg["subjects"].get(subject, [])
    return registry_get(ids[-1]) if ids else None

# Register both versions
id_v1 = registry_register("user-events-value", SCHEMA_V1)
id_v2 = registry_register("user-events-value", SCHEMA_V2)

print(f"Schema Registry:")
print(f"  v1 schema_id: {id_v1}")
print(f"  v2 schema_id: {id_v2}")
print()
print("Latest schema for 'user-events-value':")
latest = registry_latest("user-events-value")
print(f"  Fields: {[f['name'] for f in latest['fields']]}")

In [ ]:
# Simulate Kafka message format: [0x00][4-byte schema_id][avro_payload]
# Producer tags each message with its schema_id
# Consumer looks up schema from registry to deserialize

import struct

def serialize_message(schema_id, data_dict):
    """Simulate Confluent Kafka Avro message format."""
    return bytes([0x00]) + struct.pack(">I", schema_id) + pyjson.dumps(data_dict).encode()

def deserialize_message(message):
    """Simulate deserialization: look up schema, decode payload."""
    magic   = message[0]
    sid     = struct.unpack(">I", message[1:5])[0]
    payload = pyjson.loads(message[5:].decode())
    schema  = registry_get(sid)
    return sid, schema, payload

# Produce messages with schema_id embedded
sample_messages = []
for i in range(5):
    # Old producer uses v1, new producer uses v2
    if i < 3:
        msg = serialize_message(id_v1, {
            "event_id": i, "user_id": 100+i, "event_type": "click",
            "page": "/home", "revenue": 0.0,
            "event_ts": int(datetime.datetime.now().timestamp()*1000)
        })
    else:
        msg = serialize_message(id_v2, {
            "event_id": i, "user_id": 100+i, "event_type": "purchase",
            "page": "/checkout", "revenue": 99.99,
            "event_ts": int(datetime.datetime.now().timestamp()*1000),
            "device": "mobile", "country": "US", "session_id": "sess_42"
        })
    sample_messages.append(msg)

print(f"Produced {len(sample_messages)} Kafka messages")
print()

# Consumer decodes each message
print("Consuming messages:")
for msg in sample_messages:
    sid, schema, payload = deserialize_message(msg)
    fields = [f["name"] for f in schema["fields"]]
    print(f"  schema_id={sid} | fields={len(fields)} | event_type={payload['event_type']} | revenue={payload.get('revenue',0)}")

## Step 5 — Avro Pipeline: Kafka → Avro → Parquet

The standard production pattern: ingest via Avro/Kafka → store as Parquet.
This landing zone → storage zone pattern is the foundation of most data lakes.


In [ ]:
# Simulate the full pipeline
LANDING_AVRO  = f"{AVRO_DIR}/landing_zone"    # raw Avro from Kafka
STORAGE_PQ    = f"{AVRO_DIR}/storage_zone"    # cleaned Parquet for analytics
shutil.rmtree(LANDING_AVRO, ignore_errors=True)
shutil.rmtree(STORAGE_PQ, ignore_errors=True)

# Stage 1: Avro landing (simulate Kafka consumer writing Avro)
raw_v1 = spark.read.format("avro").load(AVRO_V1)
raw_v2 = spark.read.format("avro").load(AVRO_V2)

# Write to landing zone (mixed schema)
raw_v1.write.format("avro").mode("overwrite").save(f"{LANDING_AVRO}/v1")
raw_v2.write.format("avro").mode("overwrite").save(f"{LANDING_AVRO}/v2")

print("Landing zone written (raw Avro from Kafka consumer)")

# Stage 2: ETL — normalize schemas, convert to Parquet
# Fill missing fields from v1 with defaults before union
v1_normalized = raw_v1 \
    .withColumn("device",     F.lit("unknown")) \
    .withColumn("country",    F.lit("XX")) \
    .withColumn("session_id", F.lit(None).cast("string")) \
    .withColumn("ingest_date", F.current_date())

v2_normalized = raw_v2 \
    .withColumn("ingest_date", F.current_date())

combined = v1_normalized.union(v2_normalized)

# Write storage zone as Parquet (partitioned by date)
t0 = time.time()
combined.write.format("parquet") \
        .partitionBy("ingest_date") \
        .mode("overwrite") \
        .save(STORAGE_PQ)
t_etl = time.time() - t0

print(f"ETL: Avro → Parquet in {t_etl:.2f}s")
print(f"Total rows: {spark.read.parquet(STORAGE_PQ).count():,}")

# Stage 3: Analytics on Parquet
print()
print("Analytics query on Parquet storage zone:")
spark.read.parquet(STORAGE_PQ) \
     .filter(F.col("event_type") == "purchase") \
     .groupBy("country","device") \
     .agg(F.sum("revenue").alias("revenue"), F.count("*").alias("events")) \
     .orderBy(F.desc("revenue")) \
     .show(10)

## Step 6 — Avro vs Parquet Benchmark

Final comparison: write/read performance for streaming vs batch workloads.


In [ ]:
import subprocess

def size_mb(path):
    r = subprocess.run(["du","-sm",path], capture_output=True, text=True)
    return int(r.stdout.split()[0]) if r.stdout else 0

# Write benchmark
BENCH_AVRO = f"{AVRO_DIR}/bench_avro"
BENCH_PQ   = f"{AVRO_DIR}/bench_parquet"

bench_df = spark.read.format("avro").load(AVRO_V2)

print(f"{'Metric':<40} {'Avro':>12} {'Parquet':>12}")
print("-" * 68)

# Write
t0=time.time(); bench_df.write.format("avro").mode("overwrite").save(BENCH_AVRO)
t_avro_w=time.time()-t0
t0=time.time(); bench_df.write.format("parquet").mode("overwrite").save(BENCH_PQ)
t_pq_w=time.time()-t0
print(f"  {'Write time':<38} {t_avro_w:>10.3f}s {t_pq_w:>10.3f}s")

# Size
print(f"  {'Storage size (MB)':<38} {size_mb(BENCH_AVRO):>10} MB {size_mb(BENCH_PQ):>10} MB")

# Full scan
t0=time.time(); spark.read.format("avro").load(BENCH_AVRO).agg(F.sum("revenue")).collect()
t_avro_r=time.time()-t0
t0=time.time(); spark.read.parquet(BENCH_PQ).agg(F.sum("revenue")).collect()
t_pq_r=time.time()-t0
print(f"  {'Full scan (sum revenue)':<38} {t_avro_r:>10.3f}s {t_pq_r:>10.3f}s")

# Selective column read
t0=time.time(); spark.read.format("avro").load(BENCH_AVRO).select("user_id","revenue").count()
t_avro_s=time.time()-t0
t0=time.time(); spark.read.parquet(BENCH_PQ).select("user_id","revenue").count()
t_pq_s=time.time()-t0
print(f"  {'Select 2 columns':<38} {t_avro_s:>10.3f}s {t_pq_s:>10.3f}s")

print()
print("Summary:")
print(f"  Parquet is {t_avro_r/t_pq_r:.1f}x faster for full scans (columnar compression)")
print(f"  Parquet is {t_avro_s/t_pq_s:.1f}x faster for column selection (column pruning)")
print(f"  Avro storage is {'larger' if size_mb(BENCH_AVRO)>size_mb(BENCH_PQ) else 'smaller'}")
print()
print("Use Avro for Kafka/streaming. Convert to Parquet for analytics storage.")

## Summary: Avro + Schema Registry

### Avro best practices
```python
# Always write with explicit schema
df.write.format("avro")
        .option("avroSchema", json.dumps(SCHEMA))
        .save(path)

# Read with schema evolution
spark.read.format("avro")
     .option("avroSchema", json.dumps(SCHEMA_V2))   # reader schema
     .load(path_with_v1_data)                        # writer schema auto-detected
```

### Schema evolution compatibility rules
| Change | Backward | Forward | Full |
|---|---|---|---|
| Add field with default | ✅ | ❌ | ❌ |
| Remove field with default | ❌ | ✅ | ❌ |
| Add + remove with defaults | ✅ | ✅ | ✅ |
| Change type | ❌ | ❌ | ❌ |
| Rename (use alias) | ✅ | ❌ | ❌ |

### Pipeline pattern
```
Kafka → Avro (schema_id + payload) → Spark Streaming
                                        → Landing Zone (Avro)
                                        → ETL / normalize
                                        → Storage Zone (Parquet, partitioned)
                                        → Analytics (Iceberg/Delta on Parquet)
```
